# 03 — Model Comparison

Compare the Random Forest baseline and 1D CNN classifier side-by-side on
identical test sets. Evaluate statistical significance of performance differences,
analyze disagreements, and assess calibration.

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score,
    recall_score, f1_score, roc_curve,
)
from sklearn.calibration import calibration_curve

# path setup
os.chdir(os.path.join(os.path.dirname(os.path.abspath(".")), ".."))
sys.path.insert(0, os.getcwd())

from src.utils import plot_roc_pr, plot_confusion_matrix, print_metrics, save_figure
from config import VIS_CONFIG

%matplotlib inline
plt.rcParams["figure.dpi"] = VIS_CONFIG["dpi"]

## 1. Load Predictions

In [ ]:
# load saved test predictions from both models
rf_data = np.load("data/rf_test_preds.npz")
cnn_data = np.load("data/cnn_test_preds.npz")

y_true = rf_data["y_true"]
rf_probs = rf_data["y_probs"]
cnn_probs = cnn_data["y_probs"]

print(f"Test set size: {len(y_true)}")
print(f"RF predictions shape:  {rf_probs.shape}")
print(f"CNN predictions shape: {cnn_probs.shape}")
print(f"\nClass balance in test set:")
print(f"  Single (0): {(y_true == 0).sum()}")
print(f"  Binary (1): {(y_true == 1).sum()}")

## 2. ROC and PR Curves — Overlaid

In [ ]:
# overlay ROC and PR curves for both models
y_probs_dict = {"Random Forest": rf_probs, "1D CNN": cnn_probs}
fig = plot_roc_pr(y_true, y_probs_dict)
save_figure(fig, "model_comparison_roc_pr")
plt.show()

## 3. Metrics Comparison Table

In [ ]:
# compute metrics at optimal threshold (maximizing F1 on val-set-derived threshold)
def optimal_threshold(y_true, y_probs):
    """Find threshold that maximizes F1 score."""
    from sklearn.metrics import precision_recall_curve
    precision, recall, thresholds = precision_recall_curve(y_true, y_probs)
    f1_scores = 2 * precision * recall / (precision + recall + 1e-8)
    best_idx = np.argmax(f1_scores)
    return thresholds[best_idx]

results = {}
for name, probs in y_probs_dict.items():
    thresh = optimal_threshold(y_true, probs)
    preds = (probs >= thresh).astype(int)
    results[name] = {
        "Threshold": f"{thresh:.3f}",
        "Accuracy": f"{accuracy_score(y_true, preds):.4f}",
        "Precision": f"{precision_score(y_true, preds):.4f}",
        "Recall": f"{recall_score(y_true, preds):.4f}",
        "F1": f"{f1_score(y_true, preds):.4f}",
        "AUROC": f"{roc_auc_score(y_true, probs):.4f}",
    }

comparison_df = pd.DataFrame(results).T
comparison_df.index.name = "Model"
print(comparison_df.to_string())

## 4. Statistical Significance

In [ ]:
# McNemar test: are the two models making different errors?
from statsmodels.stats.contingency_tables import mcnemar

rf_thresh = optimal_threshold(y_true, rf_probs)
cnn_thresh = optimal_threshold(y_true, cnn_probs)
rf_preds = (rf_probs >= rf_thresh).astype(int)
cnn_preds = (cnn_probs >= cnn_thresh).astype(int)

rf_correct = (rf_preds == y_true)
cnn_correct = (cnn_preds == y_true)

# contingency table: [both correct, RF only correct; CNN only correct, both wrong]
a = ( rf_correct &  cnn_correct).sum()  # both correct
b = ( rf_correct & ~cnn_correct).sum()  # RF correct, CNN wrong
c = (~rf_correct &  cnn_correct).sum()  # CNN correct, RF wrong
d = (~rf_correct & ~cnn_correct).sum()  # both wrong

contingency = np.array([[a, b], [c, d]])
print("McNemar contingency table:")
print(f"  Both correct: {a},  RF only: {b}")
print(f"  CNN only: {c},  Both wrong: {d}")

result = mcnemar(contingency, exact=True)
print(f"\nMcNemar test p-value: {result.pvalue:.4e}")
print(f"Statistic: {result.statistic:.1f}")

# bootstrap DeLong test for AUROC difference
def bootstrap_auroc_diff(y_true, probs_a, probs_b, n_bootstrap=10000, seed=42):
    """Bootstrap test for difference in AUROC between two models."""
    rng = np.random.RandomState(seed)
    n = len(y_true)
    auroc_a = roc_auc_score(y_true, probs_a)
    auroc_b = roc_auc_score(y_true, probs_b)
    observed_diff = auroc_b - auroc_a

    diffs = []
    for _ in range(n_bootstrap):
        idx = rng.choice(n, size=n, replace=True)
        try:
            d = roc_auc_score(y_true[idx], probs_b[idx]) - roc_auc_score(y_true[idx], probs_a[idx])
            diffs.append(d)
        except ValueError:
            continue

    diffs = np.array(diffs)
    p_value = (np.abs(diffs - observed_diff) >= np.abs(observed_diff)).mean()
    ci_lo, ci_hi = np.percentile(diffs, [2.5, 97.5])
    return observed_diff, p_value, ci_lo, ci_hi

diff, p_val, ci_lo, ci_hi = bootstrap_auroc_diff(y_true, rf_probs, cnn_probs)
print(f"\nAUROC difference (CNN - RF): {diff:+.4f}")
print(f"95% CI: [{ci_lo:+.4f}, {ci_hi:+.4f}]")
print(f"Bootstrap p-value: {p_val:.4e}")

if p_val < 0.05:
    print("=> Statistically significant difference at alpha=0.05")
else:
    print("=> No statistically significant difference at alpha=0.05")

## 5. Error Analysis — Disagreements

In [ ]:
# categorize disagreements
cnn_only_correct = (~rf_correct &  cnn_correct)
rf_only_correct  = ( rf_correct & ~cnn_correct)
both_wrong       = (~rf_correct & ~cnn_correct)

print("Disagreement analysis:")
print(f"  CNN correct, RF wrong: {cnn_only_correct.sum()}")
print(f"  RF correct, CNN wrong: {rf_only_correct.sum()}")
print(f"  Both wrong:            {both_wrong.sum()}")
print(f"  Both correct:          {(rf_correct & cnn_correct).sum()}")

# try to load labeled sample for stellar parameter context
try:
    from astropy.table import Table
    sample = Table.read("data/labeled_sample.fits")
    test_idx = rf_data.get("test_idx", None)

    if test_idx is not None:
        test_sample = sample[test_idx]
        for cat_name, mask in [("CNN correct / RF wrong", cnn_only_correct),
                                ("RF correct / CNN wrong", rf_only_correct),
                                ("Both wrong", both_wrong)]:
            indices = np.where(mask)[0][:3]
            if len(indices) > 0:
                print(f"\n--- {cat_name} (first {len(indices)} examples) ---")
                for i in indices:
                    row = test_sample[i]
                    print(f"  APOGEE_ID={row['APOGEE_ID']}, "
                          f"Teff={row.get('TEFF', 'N/A')}, "
                          f"logg={row.get('LOGG', 'N/A')}, "
                          f"[Fe/H]={row.get('FE_H', 'N/A')}, "
                          f"true={y_true[i]}, RF_p={rf_probs[i]:.3f}, CNN_p={cnn_probs[i]:.3f}")
    else:
        print("\ntest_idx not saved in predictions file; skipping stellar param lookup.")
except Exception as e:
    print(f"\nCould not load stellar parameters: {e}")

In [ ]:
# plot example spectra from each disagreement category
try:
    spectra = np.load("data/spectra_matrix.npy")
    test_idx = rf_data.get("test_idx", None)

    if test_idx is not None:
        test_spectra = spectra[test_idx]
        categories = [
            ("CNN correct / RF wrong", cnn_only_correct),
            ("RF correct / CNN wrong", rf_only_correct),
            ("Both wrong", both_wrong),
        ]

        fig, axes = plt.subplots(3, 2, figsize=(14, 10))
        for row, (cat_name, mask) in enumerate(categories):
            indices = np.where(mask)[0]
            for col in range(2):
                ax = axes[row, col]
                if col < len(indices):
                    idx = indices[col]
                    ax.plot(test_spectra[idx], linewidth=0.4, color="k")
                    true_label = "Binary" if y_true[idx] == 1 else "Single"
                    ax.set_title(f"{cat_name}\ntrue={true_label}, RF={rf_probs[idx]:.2f}, CNN={cnn_probs[idx]:.2f}",
                                fontsize=9)
                else:
                    ax.set_visible(False)
                ax.set_ylabel("Normalized Flux")
                if row == 2:
                    ax.set_xlabel("Pixel")

        plt.tight_layout()
        save_figure(fig, "model_disagreements")
        plt.show()
    else:
        print("test_idx not available; skipping disagreement spectra plots.")
except Exception as e:
    print(f"Could not plot disagreement spectra: {e}")

## 6. Calibration

In [ ]:
# reliability diagrams (calibration curves) for both models
fig, ax = plt.subplots(figsize=(8, 7))

for name, probs in y_probs_dict.items():
    prob_true, prob_pred = calibration_curve(y_true, probs, n_bins=10, strategy="uniform")
    ax.plot(prob_pred, prob_true, "o-", label=name)

ax.plot([0, 1], [0, 1], "k--", alpha=0.4, label="Perfectly calibrated")
ax.set_xlabel("Mean predicted probability")
ax.set_ylabel("Fraction of positives")
ax.set_title("Calibration Curves (Reliability Diagrams)")
ax.legend()
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

plt.tight_layout()
save_figure(fig, "calibration_curves")
plt.show()

# assess calibration quality
from sklearn.metrics import brier_score_loss
for name, probs in y_probs_dict.items():
    brier = brier_score_loss(y_true, probs)
    print(f"{name} Brier score: {brier:.4f}")

print("\nNote: if calibration curves deviate strongly from the diagonal, "
      "consider post-hoc calibration (e.g., Platt scaling via "
      "sklearn.calibration.CalibratedClassifierCV).")

## 7. Summary

In [ ]:
# final summary
rf_auroc = roc_auc_score(y_true, rf_probs)
cnn_auroc = roc_auc_score(y_true, cnn_probs)

winner = "1D CNN" if cnn_auroc > rf_auroc else "Random Forest"
margin = abs(cnn_auroc - rf_auroc)

print("=" * 60)
print("MODEL COMPARISON SUMMARY")
print("=" * 60)
print(f"\nRandom Forest AUROC: {rf_auroc:.4f}")
print(f"1D CNN AUROC:        {cnn_auroc:.4f}")
print(f"Winner:              {winner} (+{margin:.4f} AUROC)")

# practical impact: at a fixed FPR, how many more binaries does the winner find?
from sklearn.metrics import roc_curve
fpr_rf, tpr_rf, _ = roc_curve(y_true, rf_probs)
fpr_cnn, tpr_cnn, _ = roc_curve(y_true, cnn_probs)

# compare at FPR = 5%
target_fpr = 0.05
tpr_at_fpr_rf = np.interp(target_fpr, fpr_rf, tpr_rf)
tpr_at_fpr_cnn = np.interp(target_fpr, fpr_cnn, tpr_cnn)

n_binary = (y_true == 1).sum()
extra_found = int((tpr_at_fpr_cnn - tpr_at_fpr_rf) * n_binary)

print(f"\nAt FPR = {target_fpr:.0%}:")
print(f"  RF recall:  {tpr_at_fpr_rf:.3f} ({int(tpr_at_fpr_rf * n_binary)} of {n_binary} binaries)")
print(f"  CNN recall: {tpr_at_fpr_cnn:.3f} ({int(tpr_at_fpr_cnn * n_binary)} of {n_binary} binaries)")
if extra_found > 0:
    print(f"  => CNN finds {extra_found} additional true binaries at the same false positive rate.")
elif extra_found < 0:
    print(f"  => RF finds {-extra_found} additional true binaries at the same false positive rate.")
else:
    print(f"  => Both models detect the same number of binaries at this FPR.")

print(f"\nStatistical significance: p = {p_val:.4e} (bootstrap AUROC difference)")
print("=" * 60)